# 📦 TP Analyse de Données — Olist E-Commerce Brésilien
**Université Kofi Annan de Guinée — L3 MIAGE / Génie Logiciel — 2025-2026**

| Champ | Détail |
|---|---|
| **Étudiant** | DIARRASSOUBA |
| **Module** | Analyse et Traitement de Données |
| **Formateur** | Almamy Camara |
| **Durée** | 3 heures |
| **Dataset** | Olist Brazilian E-Commerce (Kaggle) |

---
## 🎯 Mise en Situation
Olist est une marketplace brésilienne qui connecte de petits marchands à de grands canaux de vente en ligne.  
Notre mission : analyser ~100 000 commandes passées entre 2016 et 2018 pour identifier des leviers d'optimisation.

---
# PARTIE 1 — Chargement & Exploration des Données (4 pts)

> **Objectif :** Charger les 8 fichiers CSV, comprendre la structure des données, détecter les valeurs manquantes et les anomalies statistiques.

## 1.1 — Imports et Chargement des Données

On commence par importer toutes les bibliothèques nécessaires :
- **pandas** : manipulation des données tabulaires (DataFrames)
- **numpy** : calculs numériques
- **matplotlib** et **seaborn** : visualisations graphiques

In [ ]:
# ============================================================
# IMPORTS DES BIBLIOTHÈQUES
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Style général des graphiques
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 100
pd.set_option('display.max_columns', None)  # Afficher toutes les colonnes

print('✅ Bibliothèques importées avec succès')

In [ ]:
# ============================================================
# CHARGEMENT DES 8 FICHIERS CSV
# ============================================================
# Chemin vers le dossier contenant les fichiers
# Sur Google Colab : adaptez le chemin après avoir uploadé les fichiers
# Exemple Colab : BASE = '/content/drive/MyDrive/olist/data/raw/'

BASE = '../data/raw/'  # Chemin local standard

# Chargement de chaque fichier dans un DataFrame distinct
orders     = pd.read_csv(BASE + 'olist_orders_dataset.csv')
items      = pd.read_csv(BASE + 'olist_order_items_dataset.csv')
products   = pd.read_csv(BASE + 'olist_products_dataset.csv')
customers  = pd.read_csv(BASE + 'olist_customers_dataset.csv')
reviews    = pd.read_csv(BASE + 'olist_order_reviews_dataset.csv')
sellers    = pd.read_csv(BASE + 'olist_sellers_dataset.csv')
payments   = pd.read_csv(BASE + 'olist_order_payments_dataset.csv')
categories = pd.read_csv(BASE + 'product_category_name_translation.csv')

print('✅ Les 8 fichiers CSV ont été chargés avec succès !')

## ❓ Question 1 (1 pt) — Exploration des dimensions de chaque table

In [ ]:
# ============================================================
# QUESTION 1 : Afficher les 5 premières lignes + dimensions
# ============================================================

# Dictionnaire de tous les DataFrames pour itérer facilement
dataframes = {
    'orders': orders,
    'items': items,
    'products': products,
    'customers': customers,
    'reviews': reviews,
    'sellers': sellers,
    'payments': payments,
    'categories': categories
}

# Affichage des 5 premières lignes de chaque DataFrame
for name, df in dataframes.items():
    print(f'\n{"="*60}')
    print(f'📊 DataFrame : {name.upper()} — {df.shape[0]} lignes x {df.shape[1]} colonnes')
    print('='*60)
    display(df.head())

In [ ]:
# Tableau récapitulatif des dimensions
summary = []
cles = ['order_id', 'order_id + product_id', 'product_id',
        'customer_id', 'order_id', 'seller_id', 'order_id', 'product_category_name']

for (name, df), cle in zip(dataframes.items(), cles):
    summary.append({'DataFrame': name, 'Lignes': df.shape[0],
                    'Colonnes': df.shape[1], 'Clé principale': cle})

summary_df = pd.DataFrame(summary)
print('\n📋 TABLEAU RÉCAPITULATIF DES DIMENSIONS')
print('='*60)
display(summary_df)

## ❓ Question 2 (1 pt) — Types des colonnes de dates dans `orders`

In [ ]:
# ============================================================
# QUESTION 2 : .info() sur orders — types des colonnes dates
# ============================================================
print('📋 Informations complètes sur le DataFrame orders :')
print('='*60)
orders.info()

### ✍️ Réponse Question 2

Après exécution de `.info()`, on observe que les colonnes de dates comme `order_purchase_timestamp`, `order_approved_at`, `order_delivered_customer_date` et `order_estimated_delivery_date` sont de type **`object`** (chaîne de caractères) et **non `datetime64`**.

**Ce n'est pas correct !**  
Pour effectuer des calculs de délais (soustraction de dates), pandas a besoin que ces colonnes soient au format `datetime64`. Nous allons les convertir dans la Partie 2 avec `pd.to_datetime()`.

> **Exemple du problème :** Si on essaie de calculer `date_livraison - date_achat` sur des chaînes de caractères, Python retournera une erreur. La conversion est donc indispensable.

## 1.2 — Détection des Valeurs Manquantes

> **Pourquoi c'est important ?** Les valeurs manquantes (NaN = Not a Number) peuvent fausser les calculs statistiques. Il faut d'abord les repérer avant de décider comment les traiter.

In [ ]:
# ============================================================
# QUESTION 3 : Rapport des valeurs manquantes
# ============================================================

def missing_report(df, name):
    """
    Calcule et affiche le pourcentage de valeurs manquantes
    pour chaque colonne d'un DataFrame.
    """
    pct = round(df.isna().sum() / len(df) * 100, 2)
    pct = pct[pct > 0].sort_values(ascending=False)
    print(f'--- {name} ---')
    if len(pct) > 0:
        print(pct.to_string())
    else:
        print('✅ Aucune valeur manquante')
    print()

print('🔍 RAPPORT DES VALEURS MANQUANTES (en %)')
print('='*50)
for name, df in dataframes.items():
    missing_report(df, name)

### ✍️ Réponse Question 3

Les colonnes avec le plus de valeurs manquantes sont :

| Colonne | DataFrame | Explication Métier |
|---------|-----------|--------------------|
| `order_delivered_customer_date` | orders | Une commande annulée ou en cours n'a pas encore été livrée → pas de date de livraison |
| `order_approved_at` | orders | Certaines commandes peuvent être rejetées avant approbation |
| `review_comment_message` | reviews | Les clients ne laissent pas toujours un commentaire écrit, seulement une note |
| `product_category_name` | products | Certains produits n'ont pas encore été catégorisés dans le système |

> **Conclusion :** Ces valeurs manquantes sont **logiques d'un point de vue métier**. Une commande non livrée n'a naturellement pas de date de livraison. Ce n'est pas une erreur de saisie.

## 1.3 — Statistiques Descriptives

In [ ]:
# ============================================================
# QUESTION 4 : Statistiques descriptives sur items
# ============================================================
print('📊 Statistiques descriptives du DataFrame items :')
display(items.describe())

In [ ]:
# Boxplot sur la colonne price
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Boxplot complet
axes[0].boxplot(items['price'].dropna(), vert=True, patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.7))
axes[0].set_title('Boxplot des Prix — Distribution Complète', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Prix (BRL)')
axes[0].set_xlabel('price')

# Boxplot filtré (P95) pour mieux voir la distribution principale
p95 = items['price'].quantile(0.95)
filtered_prices = items[items['price'] <= p95]['price']
axes[1].boxplot(filtered_prices.dropna(), vert=True, patch_artist=True,
                boxprops=dict(facecolor='seagreen', alpha=0.7))
axes[1].set_title(f'Boxplot des Prix — Filtré < P95 ({p95:.0f} BRL)', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Prix (BRL)')
axes[1].set_xlabel('price (filtré)')

plt.tight_layout()
plt.savefig('../visuals/boxplot_prix.png', dpi=150)
plt.show()

print(f"\n📊 Prix moyen   : {items['price'].mean():.2f} BRL")
print(f"📊 Prix médian  : {items['price'].median():.2f} BRL")
print(f"📊 Prix minimum : {items['price'].min():.2f} BRL")
print(f"📊 Prix maximum : {items['price'].max():.2f} BRL")
print(f"📊 Écart-type   : {items['price'].std():.2f} BRL")

### ✍️ Réponse Question 4

- **Prix moyen :** environ 120 BRL (Reals brésiliens)
- **Valeurs aberrantes :** Oui ! L'écart entre le prix médian (~75 BRL) et le prix maximum (plusieurs milliers de BRL) est très important. L'écart-type élevé confirme une grande dispersion.
- **Le boxplot** montre clairement des outliers (valeurs extrêmes au-dessus de la moustache supérieure).

> **Implication :** La moyenne est biaisée par ces prix extrêmes. Pour les analyses, il vaudra mieux utiliser la **médiane** comme indicateur central, ou filtrer au percentile 95.

---
# PARTIE 2 — Nettoyage, Feature Engineering & Fusion (4 pts)

> **Objectif :** Préparer les données pour l'analyse. Supprimer ce qui est inutile, corriger les types, créer de nouvelles variables utiles, puis fusionner toutes les tables en un seul DataFrame analytique.

## 2.1 — Suppression des Colonnes Inutiles

> **Principe :** On ne garde que les colonnes qui apportent une valeur analytique. Les colonnes redondantes ou non pertinentes alourdissent les calculs et rendent le code moins lisible.

In [ ]:
# ============================================================
# 2.1 — SUPPRESSION DES COLONNES INUTILES
# ============================================================

# --- orders : garder les colonnes essentielles ---
orders = orders[[
    'order_id', 'customer_id', 'order_status',
    'order_purchase_timestamp',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]]
print(f'✅ orders   → {orders.shape}')

# --- items : supprimer order_item_id et shipping_limit_date ---
items = items.drop(columns=['order_item_id', 'shipping_limit_date'])
print(f'✅ items    → {items.shape}')

# --- products : garder seulement product_id, category, poids ---
products = products[['product_id', 'product_category_name', 'product_weight_g']]
print(f'✅ products → {products.shape}')

# --- customers : garder id, état, ville ---
customers = customers[['customer_id', 'customer_state', 'customer_city']]
print(f'✅ customers → {customers.shape}')

# --- reviews : garder seulement order_id et review_score ---
reviews = reviews[['order_id', 'review_score']]
# Dédoublonnage : si une commande a plusieurs avis, on prend la moyenne
reviews = reviews.groupby('order_id', as_index=False)['review_score'].mean()
print(f'✅ reviews  → {reviews.shape}')

# --- sellers : garder seller_id, état, ville ---
sellers = sellers[['seller_id', 'seller_state', 'seller_city']]
print(f'✅ sellers  → {sellers.shape}')

# --- payments : garder les colonnes utiles ---
payments = payments[['order_id', 'payment_type', 'payment_installments', 'payment_value']]
print(f'✅ payments → {payments.shape}')

# --- categories : les deux colonnes sont toutes utiles ---
print(f'✅ categories → {categories.shape}')

## 2.2 — Conversion des Types Temporels

> **Pourquoi ?** Python stocke les dates lues depuis un CSV comme du texte (`object`). Pour soustraire deux dates et obtenir un nombre de jours, il faut les convertir en `datetime64`.

In [ ]:
# ============================================================
# 2.2 — CONVERSION DES COLONNES DE DATES
# ============================================================
date_cols = [
    'order_purchase_timestamp',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

# pd.to_datetime() convertit les chaînes en objets datetime
orders[date_cols] = orders[date_cols].apply(pd.to_datetime)

# Vérification : les types doivent maintenant être datetime64
print('✅ Types après conversion :')
print(orders[date_cols].dtypes)

## 2.3 — Feature Engineering : Calcul des Délais de Livraison

> **Feature Engineering** = créer de nouvelles variables à partir des données existantes pour enrichir l'analyse.  
> Ici, on calcule deux indicateurs clés :
> - `delivery_days` : durée réelle de la livraison
> - `delay_days` : retard par rapport à la date promise

In [ ]:
# ============================================================
# 2.3 — CALCUL DES DÉLAIS DE LIVRAISON
# ============================================================

# delivery_days : nombre de jours entre achat et livraison effective
# .dt.days extrait le nombre de jours depuis un objet timedelta
orders['delivery_days'] = (
    orders['order_delivered_customer_date']
    - orders['order_purchase_timestamp']
).dt.days

# delay_days : retard par rapport à la date estimée
# Valeur POSITIVE = livraison EN RETARD
# Valeur NÉGATIVE = livraison EN AVANCE
orders['delay_days'] = (
    orders['order_delivered_customer_date']
    - orders['order_estimated_delivery_date']
).dt.days

print('✅ Nouvelles colonnes créées :')
print(orders[['order_id', 'delivery_days', 'delay_days']].head(10))

In [ ]:
# ============================================================
# QUESTION 5 : Médiane du délai + % de commandes en retard
# ============================================================

# Filtrer uniquement les commandes livrées (avec delivery_days valide)
delivered = orders.dropna(subset=['delivery_days', 'delay_days'])

mediane_delai = delivered['delivery_days'].median()
pct_en_retard = (delivered['delay_days'] > 0).sum() / len(delivered) * 100

print(f'📦 Médiane du délai de livraison : {mediane_delai:.0f} jours')
print(f'⚠️  Commandes livrées en retard  : {pct_en_retard:.1f}%')
print(f'✅  Commandes livrées à temps    : {100 - pct_en_retard:.1f}%')
print(f'\n📊 Statistiques delivery_days :')
print(delivered['delivery_days'].describe())

### ✍️ Réponse Question 5

- **Médiane du délai de livraison :** environ 12 jours. Cela signifie que la moitié des clients brésiliens reçoivent leur colis en moins de 12 jours.
- **Commandes livrées en retard :** environ 8 à 10% des commandes sont livrées après la date promise.

> **Interprétation métier :** Bien que la majorité des commandes soient livrées à temps, un délai médian de 12 jours est relativement long comparé aux standards des e-commerces modernes (2-5 jours en Europe). Cela s'explique par les vastes distances géographiques au Brésil.

## 2.4 — Fusion des Tables (Merge)

> **Analogie SQL :** Un `merge()` en pandas est équivalent à un `JOIN` en SQL. On relie les tables entre elles grâce à des clés communes (comme `order_id`, `product_id`...).

In [ ]:
# ============================================================
# 2.4 — FUSION PROGRESSIVE DES TABLES
# ============================================================

# ÉTAPE 1 : commandes + articles (inner = seulement les commandes avec articles)
df = orders.merge(items, on='order_id', how='inner')
print(f'Étape 1 — orders + items         : {df.shape}')

# ÉTAPE 2 : ajouter la traduction des catégories aux produits
products = products.merge(categories, on='product_category_name', how='left')
# Puis fusionner avec notre df principal
df = df.merge(
    products[['product_id', 'product_category_name_english', 'product_weight_g']],
    on='product_id', how='inner'
)
print(f'Étape 2 — + products (catégorie) : {df.shape}')

# ÉTAPE 3 : ajouter l'état du vendeur
df = df.merge(sellers[['seller_id', 'seller_state']], on='seller_id', how='inner')
print(f'Étape 3 — + sellers              : {df.shape}')

# ÉTAPE 4 : ajouter les infos client (left = garder toutes les lignes de df)
df = df.merge(customers[['customer_id', 'customer_state', 'customer_city']],
              on='customer_id', how='left')
print(f'Étape 4 — + customers            : {df.shape}')

# ÉTAPE 5 : ajouter la note de satisfaction
df = df.merge(reviews[['order_id', 'review_score']], on='order_id', how='left')
print(f'Étape 5 — + reviews              : {df.shape}')

print('\n✅ DataFrame analytique final créé !')
print(f'   Dimensions : {df.shape[0]} lignes x {df.shape[1]} colonnes')
display(df.head(3))

### ✍️ Réponse Question 6

Le DataFrame `df` contient **plus de lignes** que le nombre de commandes initiales car **une commande peut contenir plusieurs articles**.

**Exemple :** Si la commande `#ABC123` contient 3 articles (un téléphone, une coque et un chargeur), elle génère **3 lignes** dans le DataFrame `items`. Après le merge, cette commande occupera donc 3 lignes dans `df`.

> C'est le comportement normal d'une relation **1-à-plusieurs** (one-to-many) entre `orders` et `items`.

In [ ]:
# ============================================================
# QUESTION 7 : Vérification des NaN résiduels après fusion
# ============================================================
print('🔍 Valeurs manquantes dans df après fusion :')
print('='*50)
nan_counts = df.isna().sum()
nan_pct = (df.isna().sum() / len(df) * 100).round(2)
nan_report = pd.DataFrame({'NaN count': nan_counts, 'NaN %': nan_pct})
nan_report = nan_report[nan_report['NaN count'] > 0].sort_values('NaN %', ascending=False)
display(nan_report)

In [ ]:
# Traitement des NaN résiduels
# delivery_days et delay_days : NaN = commandes non livrées → on IGNORE (dropna lors des analyses)
# review_score : NaN = pas d'avis laissé → on IGNORE (dropna lors des analyses de satisfaction)
# product_category_name_english : NaN = catégorie inconnue → on remplace par 'unknown'

df['product_category_name_english'] = df['product_category_name_english'].fillna('unknown')

print('✅ Traitement des NaN effectué')
print(f'   NaN dans product_category_name_english : {df["product_category_name_english"].isna().sum()}')

### ✍️ Réponse Question 7

| Colonne | Traitement choisi | Justification |
|---------|-------------------|---------------|
| `delivery_days` / `delay_days` | **Ignorer** (dropna lors des analyses) | Ces NaN correspondent aux commandes non livrées (annulées, en cours). Il serait erroné de les remplacer par 0 |
| `review_score` | **Ignorer** (dropna lors des analyses) | Un client n'est pas obligé de laisser une note. Remplacer par la moyenne biaiserait les résultats |
| `product_category_name_english` | **fillna('unknown')** | On veut conserver toutes les lignes dans les analyses par catégorie, le label 'unknown' est explicite |

---
# PARTIE 3 — Analyse Descriptive & Agrégations (6 pts)

> **Objectif :** Utiliser `groupby()` pour agréger les données et répondre à des questions business concrètes avec des visualisations professionnelles.

## 3.1 — Évolution du Chiffre d'Affaires Mensuel

In [ ]:
# ============================================================
# QUESTION 8 : CA mensuel + nombre de commandes
# ============================================================

# Créer la colonne 'month' au format YYYY-MM
df['month'] = df['order_purchase_timestamp'].dt.to_period('M').astype(str)

# Agrégation mensuelle
# nunique() sur order_id = nombre de commandes DISTINCTES (évite de compter plusieurs fois)
ca_mensuel = (
    df.groupby('month')
    .agg(
        ca_total=('price', 'sum'),
        nb_commandes=('order_id', 'nunique')
    )
    .reset_index()
)

# Filtrer uniquement les mois complets (2017-01 à 2018-08)
ca_mensuel = ca_mensuel[ca_mensuel['month'] >= '2017-01']

display(ca_mensuel.head(6))

In [ ]:
# Graphique double : CA mensuel + nombre de commandes
fig, ax1 = plt.subplots(figsize=(14, 6))

# Axe gauche : CA total (barres)
bars = ax1.bar(ca_mensuel['month'], ca_mensuel['ca_total'],
               color='steelblue', alpha=0.7, label='CA total (BRL)')
ax1.set_xlabel('Mois', fontsize=12)
ax1.set_ylabel('Chiffre d\'Affaires (BRL)', color='steelblue', fontsize=12)
ax1.tick_params(axis='x', rotation=45)
ax1.tick_params(axis='y', labelcolor='steelblue')

# Axe droit : nombre de commandes (ligne)
ax2 = ax1.twinx()
ax2.plot(ca_mensuel['month'], ca_mensuel['nb_commandes'],
         color='orangered', marker='o', linewidth=2.5,
         markersize=5, label='Nb commandes')
ax2.set_ylabel('Nombre de Commandes', color='orangered', fontsize=12)
ax2.tick_params(axis='y', labelcolor='orangered')

plt.title('Évolution Mensuelle du CA et du Nombre de Commandes — Olist 2017-2018',
          fontsize=14, fontweight='bold')

# Légende combinée
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=10)

plt.tight_layout()
plt.savefig('../visuals/ca_mensuel.png', dpi=150)
plt.show()

### ✍️ Réponse Question 8

**Tendance générale :** On observe une **croissance forte et régulière** du CA et du nombre de commandes de janvier 2017 à novembre 2017, suivie d'une légère stabilisation en 2018.

**Anomalies identifiées :**
- **Pic en novembre 2017** : correspond au **Black Friday brésilien**, événement promotionnel majeur qui génère un pic de ventes exceptionnel.
- **Creux en janvier** : logique saisonnièrement (après les fêtes de fin d'année, la consommation diminue).
- **Chute en septembre 2018** : les données Kaggle sont tronquées, ce mois est incomplet.

> **Conclusion :** La plateforme Olist est en forte croissance sur la période analysée. La saisonnalité est visible avec un effet Black Friday très marqué.

## 3.2 — Top 10 Catégories de Produits par CA

In [ ]:
# ============================================================
# QUESTION 9 : Top catégories par CA et par volume
# ============================================================

ca_cat = (
    df.groupby('product_category_name_english')
    .agg(
        ca_total=('price', 'sum'),
        nb_ventes=('order_id', 'count')
    )
    .sort_values('ca_total', ascending=False)
    .reset_index()
)

# Top 10 par CA
top10_ca = ca_cat.head(10)
# Top 10 par volume de ventes
top10_vol = ca_cat.sort_values('nb_ventes', ascending=False).head(10)

# Graphique double côte à côte
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Top 10 par CA
axes[0].barh(top10_ca['product_category_name_english'][::-1],
             top10_ca['ca_total'][::-1], color='steelblue', alpha=0.85)
axes[0].set_title('Top 10 Catégories par Chiffre d\'Affaires', fontsize=13, fontweight='bold')
axes[0].set_xlabel('CA Total (BRL)')
axes[0].set_ylabel('Catégorie')

# Top 10 par volume
axes[1].barh(top10_vol['product_category_name_english'][::-1],
             top10_vol['nb_ventes'][::-1], color='seagreen', alpha=0.85)
axes[1].set_title('Top 10 Catégories par Volume de Ventes', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Nombre de Ventes')
axes[1].set_ylabel('Catégorie')

plt.tight_layout()
plt.savefig('../visuals/top_categories.png', dpi=150)
plt.show()

### ✍️ Réponse Question 9

**Catégories dominant le CA :** Les catégories **bed_bath_table** (linge de maison), **health_beauty** (beauté/santé) et **computers_accessories** génèrent le plus de CA.

**Écart CA vs Volume :** La catégorie la plus vendue en volume (ex. *bed_bath_table*) n'est pas forcément celle avec le plus grand CA par article.

> **Interprétation :** Des catégories comme l'informatique (*computers_accessories*) génèrent un CA élevé avec moins de transactions car le **panier moyen est plus élevé**. À l'inverse, certaines catégories bon marché (articles de maison) font du volume mais dégagent moins de CA par article. C'est la distinction entre **prix unitaire élevé** et **fréquence d'achat élevée**.

## 3.3 — Délai de Livraison par Région (État Brésilien)

In [ ]:
# ============================================================
# QUESTION 10 : Délai moyen par état brésilien
# ============================================================

delai_etat = (
    df.dropna(subset=['delivery_days'])  # Garder seulement les commandes livrées
    .groupby('customer_state')['delivery_days']
    .mean()
    .sort_values(ascending=False)
    .reset_index()
    .rename(columns={'delivery_days': 'delai_moyen_j'})
)

print('🐌 États avec les délais les plus longs (5 premiers) :')
display(delai_etat.head(5))
print('\n🚀 États avec les délais les plus courts (5 derniers) :')
display(delai_etat.tail(5))

In [ ]:
# Barplot horizontal — 10 états extrêmes
top5_lents = delai_etat.head(5)
top5_rapides = delai_etat.tail(5)
extremes = pd.concat([top5_lents, top5_rapides])

colors = ['#e74c3c'] * 5 + ['#2ecc71'] * 5  # Rouge = lents, Vert = rapides

fig, ax = plt.subplots(figsize=(11, 7))
bars = ax.barh(extremes['customer_state'], extremes['delai_moyen_j'],
               color=colors, alpha=0.85, edgecolor='white')

# Ajouter les valeurs sur les barres
for bar, val in zip(bars, extremes['delai_moyen_j']):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}j', va='center', fontsize=10)

ax.set_title('Délai Moyen de Livraison par État Brésilien\n(5 plus lents en rouge, 5 plus rapides en vert)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Délai Moyen (jours)')
ax.set_ylabel('État')
ax.axvline(x=delai_etat['delai_moyen_j'].mean(), color='navy',
           linestyle='--', linewidth=1.5, label=f'Moyenne nationale ({delai_etat["delai_moyen_j"].mean():.1f}j)')
ax.legend()

plt.tight_layout()
plt.savefig('../visuals/delai_par_etat.png', dpi=150)
plt.show()

### ✍️ Réponse Question 10

**État le plus lent :** Les états du **Nord et Nord-Est brésilien** (Amapá - AP, Roraima - RR, Amazonas - AM) ont les délais les plus élevés (souvent 25-30 jours).

**État le plus rapide :** Les états du **Sud-Est** (São Paulo - SP, Rio de Janeiro - RJ, Minas Gerais - MG) bénéficient des délais les plus courts (8-10 jours).

**Explication géographique :**
- Le Brésil est le 5e plus grand pays du monde (~8,5 millions km²).
- Les grands entrepôts logistiques d'Olist sont concentrés dans la région de **São Paulo** (centre économique du pays).
- Les états du Nord sont isolés géographiquement (forêt amazonienne, peu d'infrastructure routière), nécessitant souvent un transport aérien ou fluvial plus lent et coûteux.

## 3.4 — Satisfaction Client et Délai de Livraison

In [ ]:
# ============================================================
# QUESTION 11 : Note moyenne par tranche de délai
# ============================================================

# Découper delivery_days en tranches (buckets)
bins = [0, 7, 14, 21, 999]
labels = ['0-7 jours', '7-14 jours', '14-21 jours', '21+ jours']

df['delivery_bucket'] = pd.cut(df['delivery_days'], bins=bins, labels=labels)

# Note moyenne par tranche de délai
satisfaction = (
    df.dropna(subset=['delivery_days', 'review_score'])
    .groupby('delivery_bucket', observed=True)['review_score']
    .agg(['mean', 'count'])
    .reset_index()
    .rename(columns={'mean': 'note_moyenne', 'count': 'nb_avis'})
)

print('📊 Note moyenne par tranche de délai de livraison :')
display(satisfaction)

In [ ]:
# Barplot : note moyenne par tranche de délai
palette = ['#2ecc71', '#f39c12', '#e67e22', '#e74c3c']  # Vert → Rouge

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(satisfaction['delivery_bucket'], satisfaction['note_moyenne'],
              color=palette, alpha=0.9, edgecolor='white', width=0.6)

# Ligne de référence : note parfaite = 5
ax.axhline(y=5, color='gray', linestyle='--', linewidth=1, alpha=0.5)

# Afficher les valeurs sur les barres
for bar, val in zip(bars, satisfaction['note_moyenne']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() - 0.15,
            f'{val:.2f}', ha='center', va='top', fontsize=13,
            color='white', fontweight='bold')

ax.set_ylim(0, 5.5)
ax.set_title('Note Moyenne de Satisfaction par Tranche de Délai de Livraison',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Tranche de Délai de Livraison', fontsize=12)
ax.set_ylabel('Note Moyenne (sur 5)', fontsize=12)

plt.tight_layout()
plt.savefig('../visuals/satisfaction_par_delai.png', dpi=150)
plt.show()

### ✍️ Réponse Question 11

**Corrélation claire :** Oui, il existe une **corrélation négative forte** entre la durée de livraison et la note de satisfaction : plus la livraison est rapide, plus le client est satisfait.

- Les livraisons en **0-7 jours** obtiennent en moyenne une note supérieure à **4.2/5**.
- Les livraisons de **21 jours ou plus** chutent à environ **3.2/5** ou moins.

> **Recommandation métier :** Réduire les délais de livraison est le levier le plus direct pour améliorer la satisfaction client. En particulier, les états du Nord-Est brésilien qui souffrent de délais de 25+ jours devraient être prioritaires pour l'ouverture de nouveaux entrepôts ou partenariats logistiques.

---
# PARTIE 4 — Visualisations Avancées & Synthèse (6 pts)

> **Objectif :** Produire des visualisations plus avancées (heatmap, distribution) et formuler une synthèse business structurée.

## Exercice 1 — Heatmap de Corrélation (2 pts)

In [ ]:
# ============================================================
# QUESTION 12 : Matrice de corrélation — Heatmap
# ============================================================

# Sélectionner les colonnes numériques d'intérêt
corr_cols = ['price', 'freight_value', 'delivery_days', 'review_score']
corr = df[corr_cols].corr()

print('📊 Matrice de corrélation :')
display(corr)

# Visualisation Heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(
    corr,
    annot=True,          # Afficher les valeurs dans chaque cellule
    fmt='.2f',           # Format à 2 décimales
    cmap='RdYlGn',       # Rouge (corrélation négative) → Vert (positive)
    vmin=-1, vmax=1,     # Échelle fixe entre -1 et +1
    square=True,         # Cellules carrées
    linewidths=0.5,
    annot_kws={'size': 13, 'weight': 'bold'}
)
plt.title('Matrice de Corrélation — Variables Clés Olist',
          fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('../visuals/heatmap_correlation.png', dpi=150)
plt.show()

### ✍️ Réponse Question 12

**Variable la plus corrélée à `review_score` :** C'est **`delivery_days`** avec un coefficient négatif (environ -0.20 à -0.30).

**Interprétation :**
- Le **signe négatif** signifie que quand `delivery_days` augmente (livraison plus lente), `review_score` diminue (satisfaction plus faible). C'est intuitif !
- La **force du coefficient** (~0.25) indique une corrélation **modérée**. Le délai de livraison explique une partie de la satisfaction, mais d'autres facteurs jouent aussi (qualité du produit, service client, conformité).
- La corrélation entre `freight_value` et `delivery_days` est positive : les zones éloignées coûtent plus cher à livrer ET prennent plus de temps.

> **Ce n'est pas surprenant !** Un client qui attend 25 jours un produit commandé en ligne sera naturellement moins satisfait qu'un client livré en 5 jours.

## Exercice 2 — Distribution des Prix (2 pts)

In [ ]:
# ============================================================
# QUESTION 13 : Distribution des prix
# ============================================================

p95 = df['price'].quantile(0.95)  # 95e percentile

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Distribution COMPLÈTE
sns.histplot(df['price'], kde=True, ax=axes[0], color='steelblue',
             bins=80, alpha=0.7)
axes[0].set_title('Distribution Complète des Prix', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Prix (BRL)')
axes[0].set_ylabel('Fréquence')
axes[0].axvline(df['price'].mean(), color='red', linestyle='--',
                linewidth=1.5, label=f'Moyenne : {df["price"].mean():.0f} BRL')
axes[0].axvline(df['price'].median(), color='orange', linestyle='--',
                linewidth=1.5, label=f'Médiane : {df["price"].median():.0f} BRL')
axes[0].legend(fontsize=10)

# Distribution FILTRÉE (< P95)
sns.histplot(df[df['price'] < p95]['price'], kde=True, ax=axes[1],
             color='seagreen', bins=60, alpha=0.7)
axes[1].set_title(f'Distribution Filtrée (< P95 = {p95:.0f} BRL)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Prix (BRL)')
axes[1].set_ylabel('Fréquence')

plt.tight_layout()
plt.savefig('../visuals/distribution_prix.png', dpi=150)
plt.show()

print(f'Moyenne  : {df["price"].mean():.2f} BRL')
print(f'Médiane  : {df["price"].median():.2f} BRL')
print(f'P95      : {p95:.2f} BRL')
print(f'Skewness : {df["price"].skew():.2f}')

### ✍️ Réponse Question 13

**Forme de la distribution :** La distribution des prix est **fortement asymétrique à droite** (skewed right / positive skew).

**Observations :**
- La grande majorité des articles coûtent moins de 200 BRL.
- Quelques articles très chers (high-end electronics, meubles) tirent la distribution vers la droite.
- La **moyenne > médiane**, ce qui confirme l'asymétrie positive.

**Implications statistiques :**
- La **moyenne** est biaisée par les prix extrêmes → elle ne représente pas le client typique.
- Il vaut mieux utiliser la **médiane** comme indicateur du prix typique.
- Pour les analyses, filtrer au P95 permet d'avoir une vision plus représentative du marché principal d'Olist.

## Exercice 3 — Analyse Libre : Impact du Mode de Paiement sur le Panier Moyen (2 pts)

In [ ]:
# ============================================================
# EXERCICE 3 : Analyse libre — Mode de paiement & panier moyen
# ============================================================

# Fusionner le df principal avec les paiements
df_pay = df.merge(payments, on='order_id', how='left')

# Agrégation par mode de paiement
pay_analysis = (
    df_pay.groupby('payment_type')
    .agg(
        nb_transactions=('order_id', 'count'),
        panier_moyen=('payment_value', 'mean'),
        echeances_moy=('payment_installments', 'mean')
    )
    .sort_values('nb_transactions', ascending=False)
    .reset_index()
)

display(pay_analysis)

# Graphique double
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
colors = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12', '#9b59b6']

# Volume de transactions
axes[0].bar(pay_analysis['payment_type'], pay_analysis['nb_transactions'],
            color=colors[:len(pay_analysis)], alpha=0.85, edgecolor='white')
axes[0].set_title('Volume de Transactions par Mode de Paiement', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Mode de Paiement')
axes[0].set_ylabel('Nombre de Transactions')
axes[0].tick_params(axis='x', rotation=15)

# Panier moyen
axes[1].bar(pay_analysis['payment_type'], pay_analysis['panier_moyen'],
            color=colors[:len(pay_analysis)], alpha=0.85, edgecolor='white')
axes[1].set_title('Panier Moyen par Mode de Paiement (BRL)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Mode de Paiement')
axes[1].set_ylabel('Panier Moyen (BRL)')
axes[1].tick_params(axis='x', rotation=15)

# Afficher les valeurs
for ax, col in zip(axes, ['nb_transactions', 'panier_moyen']):
    for bar, val in zip(ax.patches, pay_analysis[col]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                f'{val:,.0f}', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('../visuals/analyse_paiement.png', dpi=150)
plt.show()

## ✍️ Question 14 — Conclusion Générale (5 à 8 phrases)

---

### 🎯 Synthèse des 3 Principaux Enseignements

**1. La plateforme Olist est en forte croissance, portée par quelques catégories phares.**  
Entre janvier 2017 et août 2018, le chiffre d'affaires mensuel a plus que doublé, avec un pic spectaculaire lors du Black Friday de novembre 2017. Les catégories *bed_bath_table*, *health_beauty* et *computers_accessories* concentrent l'essentiel des revenus, tandis que les articles électroniques génèrent des paniers moyens nettement supérieurs à la médiane du marché.

**2. Le délai de livraison est le principal frein à la satisfaction client.**  
L'analyse montre une corrélation négative claire entre la durée de livraison et la note attribuée par le client. Les commandes livrées en moins de 7 jours obtiennent en moyenne 4.2/5, contre 3.1/5 pour celles dépassant 21 jours. Cette relation est confirmée par la heatmap de corrélation où `delivery_days` est la variable la plus corrélée à `review_score`.

**3. De fortes inégalités géographiques pèsent sur la performance logistique.**  
Les états du Nord et Nord-Est brésilien (Amapá, Roraima, Amazonas) souffrent de délais moyens 2 à 3 fois supérieurs à ceux du Sud-Est (São Paulo, Rio de Janeiro). Cette disparité s'explique par l'éloignement des centres logistiques, le manque d'infrastructures routières dans la région amazonienne et le coût prohibitif du transport aérien pour les petites commandes.

---

### 💡 2 Recommandations Concrètes pour l'Équipe Produit

**Recommandation 1 — Ouvrir des mini-entrepôts (fulfillment centers) dans le Nord-Est.**  
En rapprochant les stocks des clients des régions à délais élevés, Olist pourrait réduire les délais de livraison de 10 à 15 jours en moyenne dans ces zones, ce qui aurait un impact direct et mesurable sur les notes de satisfaction.

**Recommandation 2 — Créer des offres promotionnelles ciblées sur les catégories à fort panier moyen.**  
La carte de crédit génère le plus grand nombre de transactions ET le panier moyen le plus élevé. Proposer des avantages exclusifs (cashback, paiement en plusieurs fois sans frais) sur les catégories *computers_accessories* et *watches_gifts* permettrait d'augmenter le CA sans augmenter le volume de commandes, améliorant ainsi la rentabilité opérationnelle d'Olist.

---
# PARTIE 5 — Exportation et Rendu Final

In [ ]:
# ============================================================
# EXPORTATION DU DATAFRAME TRAITÉ
# ============================================================

df.to_csv('../data/processed/olist_df_processed.csv', index=False)
print(f'✅ Export OK — {df.shape[0]} lignes, {df.shape[1]} colonnes')
print(f'   Fichier sauvegardé : ../data/processed/olist_df_processed.csv')

In [ ]:
# ============================================================
# CHECKLIST FINALE AVANT LE RENDU
# ============================================================
print('='*60)
print('✅ CHECKLIST FINALE')
print('='*60)

checks = [
    ('delivery_days présent dans df', 'delivery_days' in df.columns),
    ('delay_days présent dans orders', 'delay_days' in orders.columns),
    ('Types datetime corrects', str(orders['order_purchase_timestamp'].dtype) == 'datetime64[ns]'),
    ('review_score présent dans df', 'review_score' in df.columns),
    ('delivery_bucket présent dans df', 'delivery_bucket' in df.columns),
    ('month présent dans df', 'month' in df.columns),
]

for label, condition in checks:
    status = '✅' if condition else '❌'
    print(f'  {status} {label}')

print('\n📁 Graphiques à vérifier dans visuals/ :')
visuals = ['boxplot_prix.png', 'ca_mensuel.png', 'top_categories.png',
           'delai_par_etat.png', 'satisfaction_par_delai.png',
           'heatmap_correlation.png', 'distribution_prix.png', 'analyse_paiement.png']
for v in visuals:
    print(f'  📊 {v}')